In [1]:
from data.multinli_dataset import MultiNLIDataset
root_dir = "../datasets"
target_name = "gold_label_random"
confounder_names =["sentence2_has_negation"]
model_type="bert"
ds = MultiNLIDataset(root_dir,
        target_name,
        confounder_names,
        augment_data=False,
        model_type="bert",
        metadata_csv_name="metadata.csv")

In [3]:
import os, numpy as np, pandas as pd

def _split_is_numeric(series):
    return series.dtype.kind in "ifu"

def _to_txt(series):
    if _split_is_numeric(series):
        mapping = {0:'train', 1:'val', 2:'test'}
        return series.map(mapping)
    return series.astype(str)

def _to_original_format(values_txt, original_series):
    if _split_is_numeric(original_series):
        back = {'train':0, 'val':1, 'test':2}
        return values_txt.map(back).astype(int)
    return values_txt

def build_balanced_val_and_metadata_FIXED(
    csv_path,
    out_dir,
    label_col="gold_label",
    conf_col="sentence2_has_negation",
    split_col="split",
    per_group=None,
    seed=0,
    write_metadata_val_as_train=True,
    val_balanced_name="multinli_val_balanced.csv",
    meta_val_as_train_name="multinli_metadata_val_as_train.csv",
):
    os.makedirs(out_dir, exist_ok=True)
    df = pd.read_csv(csv_path, index_col=0)

    split_txt = _to_txt(df[split_col])
    if not set(split_txt.unique()) & {"train","val","test"}:
        raise ValueError("No pude interpretar 'split' (esperado 0/1/2 o train/val/test).")

    val_df = df.loc[split_txt == "val"].copy()
    if val_df.empty:
        raise ValueError("No hay filas con split == 'val'.")

    # grupos y = gold_label (0,1,2), c = sentence2_has_negation (0/1)
    y = val_df[label_col].astype(int).values
    c = val_df[conf_col].astype(int).values
    g = (y*2 + c).astype(int)

    rng = np.random.default_rng(seed)
    idx = np.arange(len(val_df))
    uniq, counts = np.unique(g, return_counts=True)
    k = counts.min() if per_group is None else int(per_group)

    chosen = []
    for gg in uniq:
        gg_idx = idx[g == gg]
        chosen.extend(rng.choice(gg_idx, size=min(len(gg_idx), k), replace=False))
    chosen = np.array(sorted(chosen))
    chosen_global_idx = val_df.iloc[chosen].index

    # guardar val balanceado (mantiene todo el DF original en esas filas)
    val_balanced_path = os.path.join(out_dir, val_balanced_name)
    df.loc[chosen_global_idx].to_csv(val_balanced_path)

    meta_val_as_train_path = None
    if write_metadata_val_as_train:
        meta_mod = df.copy()
        split_txt_full = _to_txt(meta_mod[split_col])

        # train->val (en texto), y luego elegidos val->train
        new_split_txt = split_txt_full.copy()
        new_split_txt[split_txt_full == "train"] = "val"
        new_split_txt.loc[new_split_txt.index.isin(chosen_global_idx)] = "train"

        # convertir de vuelta al formato original
        meta_mod[split_col] = _to_original_format(new_split_txt, df[split_col])

        meta_val_as_train_path = os.path.join(out_dir, meta_val_as_train_name)
        meta_mod.to_csv(meta_val_as_train_path)

    return val_balanced_path, meta_val_as_train_path



In [4]:
csv_path = "/workspace1/araymond/datasets/multinli/data/metadata.csv"
out_dir  = "/workspace1/araymond/datasets/multinli/data"

val_balanced, meta_val_as_train = build_balanced_val_and_metadata_FIXED(
    csv_path, out_dir, per_group=None, seed=0, write_metadata_val_as_train=True
)
print(val_balanced)
print(meta_val_as_train)



/workspace1/araymond/datasets/multinli/data/multinli_val_balanced.csv
/workspace1/araymond/datasets/multinli/data/multinli_metadata_val_as_train.csv


In [5]:
import pandas as pd

def split_to_txt(s):
    m = {0:'train', 1:'val', 2:'test', '0':'train','1':'val','2':'test'}
    return s.map(lambda x: m.get(x, str(x).lower()))

def check_multinli(csv_path):
    df = pd.read_csv(csv_path, index_col=0)

    # normaliza split a texto para ver tablas legibles
    split_txt = split_to_txt(df['split'])

    # y: 0/1/2, c: 0/1
    y = df['gold_label'].astype(int)
    c = df['sentence2_has_negation'].astype(int)

    # Conteo total por grupo (y, c)
    print("\n== Conteo total por (gold_label, negation) ==")
    print(pd.crosstab(y, c, rownames=['gold_label'], colnames=['negation']))

    # Conteo por split y grupo
    print("\n== Conteo por split y grupo ==")
    table = pd.crosstab([split_txt, y], c, rownames=['split','gold_label'], colnames=['negation'])
    print(table)

    # Si quieres el id de grupo (y*2 + c) y su conteo:
    gid = (y*2 + c).astype(int)
    print("\n== Conteo por group_id (y*2+c) ==")
    print(gid.value_counts().sort_index())

# Ejemplos:
check_multinli("../datasets/multinli/data/multinli_val_balanced.csv")
check_multinli("../datasets/multinli/data/multinli_metadata_val_as_train.csv")



== Conteo total por (gold_label, negation) ==
negation      0    1
gold_label          
0           613  613
1           613  613
2           613  613

== Conteo por split y grupo ==
negation            0    1
split gold_label          
val   0           613  613
      1           613  613
      2           613  613

== Conteo por group_id (y*2+c) ==
0    613
1    613
2    613
3    613
4    613
5    613
Name: count, dtype: int64

== Conteo total por (gold_label, negation) ==
negation         0      1
gold_label               
0           114909  22447
1           134821   3020
2           133215   3937

== Conteo por split y grupo ==
negation              0      1
split gold_label              
test  0           34597   6655
      1           40496    886
      2           39930   1148
train 0             613    613
      1             613    613
      2             613    613
val   0           79699  15179
      1           93712   1521
      2           92672   2176

== Conteo por g